In [1]:
#!/usr/bin/python3

#Code for gradient evaluation
#using the QNDM method.
#Written by: G. Minuto and S. Caletti
#Contacts: giovanni.minuto@uniroma1.it
#Cite 2301.07128 [quant-ph] in case you use 
#these code or part of it.

#--------------------------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np
from math import pi
import random

#---------------------------------------------------------------------------------------------
#import QNDM package

from qndm.hamiltonians.examples import get_SparsePauliOp
from qndm.core import *
from qndm.hamiltonians.examples import get_hamiltonian
from qndm.hamiltonians.hydrogen import get_model
from qndm.utils.error import get_dm_error
from qndm.core_ham_learning import distribution_before_after_detector_measurament


#---------------------------------------------------------------------------------------------

SEED = 43

# Set global seeds
random.seed(SEED)
np.random.seed(SEED)

############################
#                          #
#      Hamiltonian M       #
#                          #
############################

hamlib_ = False

if hamlib_ == True:
    # Input to use with the qndm.hamiltonians.hydrogen package
    n = 2 # atoms count
    shape = 'linear' # linear, pyramid, ring, sheet
    r = 0.6 # between 0.5 and 2.0 (step 0.1)
    key = "/ham_BK/"
    
    #get the model (we need for first because sets nqubit)
    model = get_model(n, shape, r, key)

    PS = model["PS"]
    cps = model["cps"]
    pauli_string = len(cps)
    num_qub = model["nqubit"]
  

else:

    #number of qubit of the quantum register
    num_qub = 2

    #select the pauli string number
    pauli_string = 2

    #hamiltonians M
    PS, cps = get_hamiltonian(num_qub, pauli_string,sel=10, mu=5, sigma=0.1)

print(PS)


spop = get_SparsePauliOp(PS, cps) #spop = sparse pauli operator


############################
#                          #
#     Quantum Circuit      #
#                          #
############################


#layer = rotational layers + entanglement layer
#number of layers = rotational layers + entanglement layer
num_l = 4

#inside a layer: number of rotational layers
lay_u = 1

#entanglement layer
ent_gate = 0
# if ent_gate = 0 ---> CNOT
# if ent_gate = 1 ---> SWAP

#total number of parameters per qubit
n_pars=lay_u*num_l*num_qub 

#Rotational array: here there are the gates information to implent unitary trasformation U
#code: rx = 1, ry = 2, rz = 3
val_g = np.random.randint(1, 4, size=n_pars) #val_g = [1,1,2,2,3,3]

#Parameters array: here there are the parameters information for each gates in U
pars = np.random.rand(n_pars)



#############################
#                           #
#  Derivative Paramenters   #
#                           #
#############################


# Input to use with the qndm.hamiltonians.example package

shots = 500000 #number shots for a single evaluation

#shift (paramenter shift rule)
shift = 0.1



['XZ', 'IZ', 'XI', 'IZ', 'II', 'XI', 'YX', 'YI', 'ZY', 'XY']


In [2]:
#!/usr/bin/python3

#Code for gradient evaluation
#using the QNDM method.
#Written by: G. Minuto and S. Caletti
#Contacts: giovanni.minuto@uniroma1.it
#Cite 2301.07128 [quant-ph] in case you use 
#these code or part of it.

#--------------------------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np
from math import pi
import random





#---------------------------------------------------------------------------------------------
#import QNDM package

from qndm.hamiltonians.examples import add_detector, get_SparsePauliOp
from qndm.core import *
from qndm.hamiltonians.examples import get_hamiltonian
from qndm.hamiltonians.hydrogen import get_model
from qndm.utils.error import get_qndm_error






#hamiltonians for QNDM: here we add the detector operator equal to Z
PS_QNDM, cps_QNDM = add_detector(PS, cps)
newspop = get_SparsePauliOp(PS_QNDM, cps_QNDM) #After adding the detector



#############################
#                           #
#  Derivative Paramenters   #
#                           #
#############################


# Input to use with the qndm.hamiltonians.example package

shots = 500000 #number shots for a single evaluation

#shift (paramenter shift rule)
shift = 0.1


#coupling parameter QNDM
lambda1 = 0.9

#--------------------------------------------------------------------------------------------
#R U N - C A R D#

#print_run_card(output_dir="./output_test", n_qubits = num_qub, n_layers = num_l, val_g = val_g, spop =  newspop, n_shots = shots, lambda1 = lambda1, ent_gate=0,)
#------------------------------------------------------------------


print("Into the derivatives process...", end="")

#gradient with qndm
before,after = distribution_before_after_detector_measurament(lambda1=lambda1, 
              pars=pars,
              newspop = newspop,
              num_qub = num_qub,
              num_l = num_l,
              ent_gate = ent_gate,
              shift = 0.1,            
              shots = shots,
              val_g = val_g)


print(before)
print(after)
final = 0
for key, value in after.items(): 
    final += np.abs(after[key]-before[key]) 
print(final)







Into the derivatives process...ciao
0.1
Circ one
     ┌─────────────┐     ┌──────────────┐     ┌─────────────┐     »
q_0: ┤ Rx(0.56609) ├──■──┤ Ry(0.029014) ├──■──┤ Rz(0.39495) ├──■──»
     ├─────────────┤┌─┴─┐├─────────────┬┘┌─┴─┐├─────────────┤┌─┴─┐»
q_1: ┤ Rx(0.54116) ├┤ X ├┤ Ry(0.73375) ├─┤ X ├┤ Rx(0.80205) ├┤ X ├»
     └────┬───┬────┘└───┘└─────────────┘ └───┘└─────────────┘└───┘»
q_2: ─────┤ H ├───────────────────────────────────────────────────»
          └───┘                                                   »
c: 3/═════════════════════════════════════════════════════════════»
                                                                  »
«     ┌─────────────┐      »
«q_0: ┤ Ry(0.25442) ├───■──»
«     ├─────────────┴┐┌─┴─┐»
«q_1: ┤ Rz(0.056885) ├┤ X ├»
«     └──────────────┘└───┘»
«q_2: ─────────────────────»
«                          »
«c: 3/═════════════════════»
«                          »
«     ┌───────────────────────────────────────────────────────────────────────

KeyError: '110'